# Prefix prompt — bert-base-multilingual-cased

Part of the M.S. thesis *Hate Speech Classification in Roman Urdu: A Comparative Study on Parameter Efficient Fine-Tuning and Prompt Engineering*.

Few-shot **prefix-style prompt-based learning** of `bert-base-multilingual-cased` with [OpenPrompt](https://github.com/thunlp/OpenPrompt), using the Roman Urdu template `<comment> Ye hai [MASK]!` and a Roman Urdu verbalizer. Trained with K = 32 to 256 examples per class over 2 random rounds; the best model is selected by macro F1.

**Setup:** install dependencies with `pip install -r requirements.txt` from the repository root and place the dataset in `data/` (see the main README).

In [ ]:
import os

# Paths are relative to this notebook. See data/README.md for the dataset.
DATA_PATH = "../../data/PURRUTT.xlsx"
SPECIAL_EXAMPLES_PATH = "../../data/special_examples.csv"
OUTPUT_DIR = "../../outputs/prefix_prompt_mbert"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
from openprompt.data_utils import InputExample
from openprompt.plms import load_plm
from openprompt.prompts import ManualTemplate, ManualVerbalizer
from openprompt import PromptForClassification, PromptDataLoader
from transformers import Trainer, TrainingArguments
import pandas as pd
import torch
from evaluate import load as load_metric
import numpy as np
from torch.utils.data import Dataset
import os
import random
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
import json
import shutil
import sys
import torch
from torch import nn
from torch.nn.parallel.data_parallel import DataParallel
from openprompt.utils.cuda import model_to_device
from tensorboardX import SummaryWriter
from tqdm import tqdm
import dill
import warnings
from typing import Callable, Union, Dict, Optional, List
from collections import OrderedDict
from openprompt.utils.logging import logger

In [ ]:
# Load the dataset
dataset_path = DATA_PATH
df = pd.read_excel(dataset_path)
df = df.dropna(subset=['Toxic']) # Drop rows where 'Toxic' is NaN

# Map the dataset to InputExamples
dataset = []

# Create dataset and mapping
guid_to_text = {}

for index, row in df.iterrows():
    example = InputExample(guid=index, text_a=row['Comment'], label=int(row['Toxic']))
    dataset.append(example)
    guid_to_text[index] = row['Comment']  # Store the guid and text_a


# Print the first 5 examples to verify
for example in dataset[:5]:
    print(vars(example))

In [ ]:
# Define a function to create few-shot datasets
def create_few_shot_datasets(dataset, K, num_classes):
    few_shot_train_examples = []
    few_shot_dev_examples = []
    remaining_examples = []

    for class_label in range(num_classes):
        class_examples = [example for example in dataset if example.label == class_label]
        if len(class_examples) < K:
            print(f"Not enough examples for class {class_label}. Required: {K}, Available: {len(class_examples)}")
            return [], [], []

        selected_examples = random.sample(class_examples, K)
        few_shot_train_examples.extend(selected_examples)
        few_shot_dev_examples.extend(selected_examples)
        remaining_examples.extend([example for example in class_examples if example not in selected_examples])

    return few_shot_train_examples, few_shot_dev_examples, remaining_examples

In [ ]:
# Define the prompt template with the specified text
plm, tokenizer, model_config, WrapperClass = load_plm("bert", "bert-base-multilingual-cased")

prompt_template = ManualTemplate(
    text='{"placeholder":"text_a"} Ye hai {"mask"}!',
    tokenizer=tokenizer,
)


# Define a verbalizer for binary classification with Roman Urdu words
verbalizer = ManualVerbalizer(
    classes=["0", "1"],
    label_words={
        "0": ["muhabbat", "pyaar", "acha", "dua", "wah"],  # Non-toxic words
        "1": ["khilaf", "nafrat", "ghussa", "lanat", "ganda","behnchod", "gaali", "bura"],  # Toxic words
    },
    tokenizer=tokenizer,
)

In [ ]:
# Define a configuration object (can be customized)
from yacs.config import CfgNode as CN

config = CN()
config.train = CN()
config.train.num_epochs = 3
config.train.gradient_accumulation_steps = 1
config.train.max_grad_norm = 1.0
config.train.learning_rate = 5e-5

config.logging = CN()
config.logging.path = "./logs"

config.environment = CN()
config.environment.local_rank = 0

# Number of classes
num_classes = 2

# Different K values to test
K_values = [ 32,64,128]

# Initialize the directory for saving models
save_path = os.path.join(OUTPUT_DIR, "saved_models")
if not os.path.exists(save_path):
    os.makedirs(save_path)

In [ ]:
import pandas as pd
import os
import json

# Initialize variables to store predictions and metrics
train_results = []
test_results = []
all_metrics = []

best_model_metrics = None  # To track the best model metrics
best_model_path = None  # To track the path of the best model

num_rounds = 2

for K in K_values:
    for round_num in range(num_rounds):
        # Split the dataset into training, development, and test sets
        train_examples, dev_examples, remaining_examples = create_few_shot_datasets(dataset, K, num_classes)
        test_examples = remaining_examples

        # Create data loaders
        train_dataloader = PromptDataLoader(
            dataset=train_examples,
            template=prompt_template,
            tokenizer=tokenizer,
            tokenizer_wrapper_class=WrapperClass,
            max_seq_length=128,
            batch_size=8,
            shuffle=True,
            teacher_forcing=False,
            predict_eos_token=False,
            truncate_method="head"
        )

        dev_dataloader = PromptDataLoader(
            dataset=dev_examples,
            template=prompt_template,
            tokenizer=tokenizer,
            tokenizer_wrapper_class=WrapperClass,
            max_seq_length=128,
            batch_size=8,
            shuffle=False,
            teacher_forcing=False,
            predict_eos_token=False,
            truncate_method="head"
        )

        test_dataloader = PromptDataLoader(
            dataset=test_examples,
            template=prompt_template,
            tokenizer=tokenizer,
            tokenizer_wrapper_class=WrapperClass,
            max_seq_length=128,
            batch_size=8,
            shuffle=False,
            teacher_forcing=False,
            predict_eos_token=False,
            truncate_method="head"
        )

        # Define the model for classification
        model = PromptForClassification(
            template=prompt_template,
            plm=plm,
            verbalizer=verbalizer,
        )
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)

        # Optimizer
        optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
        num_epochs = 3

        for epoch in range(num_epochs):
            model.train()
            total_loss = 0
            for batch in train_dataloader:
                batch = batch.to(device)
                labels = batch['label'].to(device)
                optimizer.zero_grad()
                logits = model(batch)
                loss = torch.nn.CrossEntropyLoss()(logits, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_dataloader)}")

        # Evaluate the model
        model.eval()
        all_labels = []
        all_preds = []
        with torch.no_grad():
            for batch in test_dataloader:
                batch = batch.to(device)
                labels = batch['label'].to(device)
                logits = model(batch)
                preds = torch.argmax(logits, dim=-1)
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())

        # Compute metrics
        precision = precision_score(all_labels, all_preds, average='macro')
        recall = recall_score(all_labels, all_preds, average='macro')
        f1 = f1_score(all_labels, all_preds, average='macro')
        accuracy = accuracy_score(all_labels, all_preds)

        # Log metrics
        metrics = {
            'K': K,
            'round': round_num + 1,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'accuracy': accuracy,
        }
        all_metrics.append(metrics)

        print(f"K={K}, Round={round_num + 1}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}, Accuracy: {accuracy:.4f}")

        # Save the best model based on F1 score
        model_save_path = os.path.join(save_path, f"model_K_{K}_round_{round_num + 1}.pth")
        if best_model_metrics is None or f1 > best_model_metrics['f1_score']:
            best_model_metrics = metrics
            best_model_path = model_save_path
            torch.save(model.state_dict(), best_model_path)

# Load the best model for evaluation on special examples
print(f"Best model: {best_model_path} with F1 Score: {best_model_metrics['f1_score']:.4f}")
model.load_state_dict(torch.load(best_model_path))
model.eval()

# # Evaluate the best model on special examples
# special_examples_path = SPECIAL_EXAMPLES_PATH
# special_df = pd.read_csv(special_examples_path)

# special_results = []
# with torch.no_grad():
#     for index, row in special_df.iterrows():
#         example = InputExample(guid=index, text_a=row['Comment'], label=row['Toxic'])
#         dataloader = PromptDataLoader(
#             dataset=[example],
#             template=prompt_template,
#             tokenizer=tokenizer,
#             tokenizer_wrapper_class=WrapperClass,
#             max_seq_length=128,
#             batch_size=1,
#             shuffle=False,
#             teacher_forcing=False,
#             predict_eos_token=False,
#             truncate_method="head"
#         )

#         for batch in dataloader:
#             batch = batch.to(device)
#             logits = model(batch)
#             preds = torch.argmax(logits, dim=-1).item()
#             special_results.append({
#                 "comment": row['Comment'],
#                 "ground_label": row['Toxic'],
#                 "predicted_label": preds
#             })

# # Update the CSV with predictions
# special_df['Predicted_Toxic'] = [result['predicted_label'] for result in special_results]
# special_df.to_csv(SPECIAL_EXAMPLES_PATH, index=False)

# # Print the results for special examples
# print("Special Examples Evaluation Results:")
# for result in special_results:
#     print(f"Comment: {result['comment']}, Ground Label: {result['ground_label']}, Predicted Label: {result['predicted_label']}")

In [ ]:
import pandas as pd
import os
import json

# Initialize variables to store predictions and metrics
train_results = []
test_results = []
all_metrics = []

best_model_metrics = None  # To track the best model metrics
best_model_path = None  # To track the path of the best model

num_rounds = 2
K_values = [128,256]

for K in K_values:
    for round_num in range(num_rounds):
        # Split the dataset into training, development, and test sets
        train_examples, dev_examples, remaining_examples = create_few_shot_datasets(dataset, K, num_classes)
        test_examples = remaining_examples

        # Create data loaders
        train_dataloader = PromptDataLoader(
            dataset=train_examples,
            template=prompt_template,
            tokenizer=tokenizer,
            tokenizer_wrapper_class=WrapperClass,
            max_seq_length=128,
            batch_size=8,
            shuffle=True,
            teacher_forcing=False,
            predict_eos_token=False,
            truncate_method="head"
        )

        dev_dataloader = PromptDataLoader(
            dataset=dev_examples,
            template=prompt_template,
            tokenizer=tokenizer,
            tokenizer_wrapper_class=WrapperClass,
            max_seq_length=128,
            batch_size=8,
            shuffle=False,
            teacher_forcing=False,
            predict_eos_token=False,
            truncate_method="head"
        )

        test_dataloader = PromptDataLoader(
            dataset=test_examples,
            template=prompt_template,
            tokenizer=tokenizer,
            tokenizer_wrapper_class=WrapperClass,
            max_seq_length=128,
            batch_size=8,
            shuffle=False,
            teacher_forcing=False,
            predict_eos_token=False,
            truncate_method="head"
        )

        best_model_path = os.path.join(OUTPUT_DIR, "saved_models", "model_K_64_round_1.pth")

        # Load the best model
        model = PromptForClassification(
            template=prompt_template,
            plm=plm,
            verbalizer=verbalizer,
        )
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)
        model.load_state_dict(torch.load(best_model_path))



        # Optimizer
        optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
        num_epochs = 3

        for epoch in range(num_epochs):
            model.train()
            total_loss = 0
            for batch in train_dataloader:
                batch = batch.to(device)
                labels = batch['label'].to(device)
                optimizer.zero_grad()
                logits = model(batch)
                loss = torch.nn.CrossEntropyLoss()(logits, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_dataloader)}")

        # Evaluate the model
        model.eval()
        all_labels = []
        all_preds = []
        with torch.no_grad():
            for batch in test_dataloader:
                batch = batch.to(device)
                labels = batch['label'].to(device)
                logits = model(batch)
                preds = torch.argmax(logits, dim=-1)
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())

        # Compute metrics
        precision = precision_score(all_labels, all_preds, average='macro')
        recall = recall_score(all_labels, all_preds, average='macro')
        f1 = f1_score(all_labels, all_preds, average='macro')
        accuracy = accuracy_score(all_labels, all_preds)

        # Log metrics
        metrics = {
            'K': K,
            'round': round_num + 1,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'accuracy': accuracy,
        }
        all_metrics.append(metrics)

        print(f"K={K}, Round={round_num + 1}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}, Accuracy: {accuracy:.4f}")

        # Save the best model based on F1 score
        model_save_path = os.path.join(save_path, f"model_K_{K}_round_{round_num + 1}.pth")
        if best_model_metrics is None or f1 > best_model_metrics['f1_score']:
            best_model_metrics = metrics
            best_model_path = model_save_path
            torch.save(model.state_dict(), best_model_path)

# Load the best model for evaluation on special examples
print(f"Best model: {best_model_path} with F1 Score: {best_model_metrics['f1_score']:.4f}")
model.load_state_dict(torch.load(best_model_path))
model.eval()

# # Evaluate the best model on special examples
# special_examples_path = SPECIAL_EXAMPLES_PATH
# special_df = pd.read_csv(special_examples_path)

# special_results = []
# with torch.no_grad():
#     for index, row in special_df.iterrows():
#         example = InputExample(guid=index, text_a=row['Comment'], label=row['Toxic'])
#         dataloader = PromptDataLoader(
#             dataset=[example],
#             template=prompt_template,
#             tokenizer=tokenizer,
#             tokenizer_wrapper_class=WrapperClass,
#             max_seq_length=128,
#             batch_size=1,
#             shuffle=False,
#             teacher_forcing=False,
#             predict_eos_token=False,
#             truncate_method="head"
#         )

#         for batch in dataloader:
#             batch = batch.to(device)
#             logits = model(batch)
#             preds = torch.argmax(logits, dim=-1).item()
#             special_results.append({
#                 "comment": row['Comment'],
#                 "ground_label": row['Toxic'],
#                 "predicted_label": preds
#             })

# # Update the CSV with predictions
# special_df['Predicted_Toxic'] = [result['predicted_label'] for result in special_results]
# special_df.to_csv(SPECIAL_EXAMPLES_PATH, index=False)

# # Print the results for special examples
# print("Special Examples Evaluation Results:")
# for result in special_results:
#     print(f"Comment: {result['comment']}, Ground Label: {result['ground_label']}, Predicted Label: {result['predicted_label']}")

In [ ]:
import torch
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
train_examples, dev_examples, remaining_examples = create_few_shot_datasets(dataset, 2, 2)
test_examples = remaining_examples
# Define the path to the saved best model
best_model_path = os.path.join(OUTPUT_DIR, "saved_models", "model_K_128_round_2.pth")

# Load the best model
model = PromptForClassification(
    template=prompt_template,
    plm=plm,
    verbalizer=verbalizer,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.load_state_dict(torch.load(best_model_path))
model.eval()

# Prepare the test dataset
test_dataloader = PromptDataLoader(
    dataset=test_examples,  # Replace with your test examples
    template=prompt_template,
    tokenizer=tokenizer,
    tokenizer_wrapper_class=WrapperClass,
    max_seq_length=128,
    batch_size=8,
    shuffle=False,
    teacher_forcing=False,
    predict_eos_token=False,
    truncate_method="head"
)

# Evaluate the model on the test dataset
test_results = []
all_labels = []
all_preds = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = batch.to(device)
        labels = batch['label'].to(device)  # Ground truth labels
        logits = model(batch)  # Model predictions
        preds = torch.argmax(logits, dim=-1)  # Predicted labels

        # Collect predictions and ground truth
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

        # Store the test results
        for i, guid in enumerate(batch['guid']):
            guid = guid.item()  # Convert tensor to integer
            test_results.append({
                "comment": guid_to_text[guid],  # Original text using guid
                "ground_label": labels[i].item(),
                "predicted_label": preds[i].item()
            })

# Compute metrics
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')
accuracy = accuracy_score(all_labels, all_preds)

# Print test set metrics
print("\nTest Set Metrics:")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}, Accuracy: {accuracy:.4f}")

# Print test results
print("\nTest Set Results:")
for result in test_results:
    print(f"Comment: {result['comment']}, Ground Label: {result['ground_label']}, Predicted Label: {result['predicted_label']}")